In [ ]:
import os
import psycopg2
from dotenv import load_dotenv
from pgvector.psycopg2 import register_vector
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from langchain_openai import ChatOpenAI

load_dotenv()

In [ ]:
client = OpenAI()

In [ ]:
conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST"),
    database=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD")
)

In [ ]:
register_vector(conn)
cur = conn.cursor()

In [ ]:
# Load resume
loader = PyPDFLoader("resume.pdf")
docs = loader.load()

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

In [ ]:
conn.autocommit = True

In [ ]:
for i, chunk in enumerate(chunks, 1):
    text = chunk.page_content
    try:
        emb = client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        ).data[0].embedding
        
        cur.execute(
            "INSERT INTO resume_embeddings (chunk, embedding) VALUES (%s, %s)",
            (text, emb)
        )
        print(f"Inserted chunk {i}/{len(chunks)}")
    except Exception as e:
        print(f"Skipped chunk {i} due to error: {e}")
        conn.rollback()  